In [0]:
# MVP Engenharia de Dados
# ETAPA 03 - Camada Gold e análise das variáveis associadas à nota final

from pyspark.sql import functions as F

SILVER_TABLE = "workspace.mvp_silver.student_productivity_clean"
GOLD_TABLE = "workspace.mvp_gold.student_performance"

df = spark.table(SILVER_TABLE)

print("=== SILVER CARREGADA ===")
print("Registros:", df.count())
print("Colunas:", len(df.columns))

=== SILVER CARREGADA ===
Registros: 5999
Colunas: 21


In [0]:
# Seleção das variáveis relevantes para a análise

analysis_cols = [
    "student_id",
    "age",
    "gender",
    "study_hours_per_day",
    "sleep_hours",
    "phone_usage_hours",
    "social_media_hours",
    "youtube_hours",
    "gaming_hours",
    "entertainment_hours",
    "breaks_per_day",
    "coffee_intake_mg",
    "exercise_minutes",
    "assignments_completed",
    "attendance_percentage",
    "stress_level",
    "focus_score",
    "productivity_score",
    "final_grade"
]

df_gold = df.select(*analysis_cols)

print("=== GOLD PREPARADA ===")
print("Registros:", df_gold.count())
print("Colunas:", len(df_gold.columns))

display(df_gold.limit(10))

=== GOLD PREPARADA ===
Registros: 5999
Colunas: 19


student_id,age,gender,study_hours_per_day,sleep_hours,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,entertainment_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,productivity_score,final_grade
1,23,Female,4.35,3.63,3.38,2.73,1.83,5.26,9.82,6,347,111,2,57.21,10,57,33.78,81.87
2,20,Male,6.14,6.58,5.48,1.51,3.13,1.73,6.37,13,403,28,10,91.27,10,49,48.99,60.9
3,29,Female,4.98,3.26,4.83,3.63,0.18,4.71,8.52,1,419,102,8,63.14,2,38,36.6,86.22
4,27,Female,3.19,4.58,10.06,3.95,5.75,2.52,12.22,9,178,28,18,40.51,6,50,19.87,71.77
5,24,Male,7.67,6.21,3.02,1.59,5.46,5.65,12.7,8,436,105,7,45.53,6,41,52.9,90.13
6,29,Other,7.18,3.52,4.02,3.74,1.42,0.16,5.32,10,392,12,3,47.58,10,70,47.31,59.48
7,21,Female,9.06,6.36,11.45,5.99,2.2,4.44,12.63,14,87,28,15,43.5,8,35,41.23,62.71
8,23,Female,6.37,4.86,3.31,1.37,4.36,5.13,10.86,2,152,103,17,75.22,6,59,53.81,52.22
9,26,Male,4.19,4.87,9.66,2.87,0.1,3.38,6.35,13,460,42,11,44.79,3,39,25.99,76.15
10,19,Female,7.28,9.56,2.13,0.81,1.35,2.55,4.71,7,416,107,6,79.15,10,73,73.18,88.53


In [0]:
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print("Tabela Gold persistida com sucesso:")
print(GOLD_TABLE)

Tabela Gold persistida com sucesso:
workspace.mvp_gold.student_performance


In [0]:
# Correlação das variáveis numéricas com a nota final

variaveis = [
    "age",
    "study_hours_per_day",
    "sleep_hours",
    "phone_usage_hours",
    "social_media_hours",
    "youtube_hours",
    "gaming_hours",
    "entertainment_hours",
    "breaks_per_day",
    "coffee_intake_mg",
    "exercise_minutes",
    "assignments_completed",
    "attendance_percentage",
    "stress_level",
    "focus_score",
    "productivity_score"
]

resultados = []

for variavel in variaveis:
    correlacao = df_gold.stat.corr(
        variavel,
        "final_grade"
    )

    resultados.append(
        (
            variavel,
            float(correlacao),
            abs(float(correlacao))
        )
    )

df_correlacoes = spark.createDataFrame(
    resultados,
    [
        "variavel",
        "correlacao_final_grade",
        "correlacao_absoluta"
    ]
)

df_correlacoes = df_correlacoes.orderBy(
    F.desc("correlacao_absoluta")
)

display(df_correlacoes)

variavel,correlacao_final_grade,correlacao_absoluta
youtube_hours,-0.03789777416336326,0.03789777416336326
gaming_hours,0.033186398180796835,0.033186398180796835
stress_level,-0.028849798500437788,0.028849798500437788
study_hours_per_day,-0.026335637303128864,0.026335637303128864
sleep_hours,0.021743102746073755,0.021743102746073755
attendance_percentage,-0.01878119588133075,0.01878119588133075
assignments_completed,0.015354137823888492,0.015354137823888492
productivity_score,-0.012625055151133095,0.012625055151133095
focus_score,-0.009798536785877264,0.009798536785877264
age,0.009400385739671082,0.009400385739671082


In [0]:
# Ranking final das associações com final_grade
# Valores arredondados para facilitar interpretação e documentação

df_ranking = (
    df_correlacoes
    .select(
        "variavel",
        F.round("correlacao_final_grade", 4).alias("correlacao"),
        F.round("correlacao_absoluta", 4).alias("forca_associacao")
    )
    .orderBy(F.desc("forca_associacao"))
)

display(df_ranking)

variavel,correlacao,forca_associacao
youtube_hours,-0.0379,0.0379
gaming_hours,0.0332,0.0332
stress_level,-0.0288,0.0288
study_hours_per_day,-0.0263,0.0263
sleep_hours,0.0217,0.0217
attendance_percentage,-0.0188,0.0188
assignments_completed,0.0154,0.0154
productivity_score,-0.0126,0.0126
focus_score,-0.0098,0.0098
age,0.0094,0.0094


In [0]:
# Persistência do ranking de correlações na camada Gold

CORRELATION_TABLE = (
    "workspace.mvp_gold.correlation_with_final_grade"
)

(
    df_ranking.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CORRELATION_TABLE)
)

print("Tabela de correlações persistida com sucesso:")
print(CORRELATION_TABLE)

Tabela de correlações persistida com sucesso:
workspace.mvp_gold.correlation_with_final_grade


In [0]:
# Resumo das associações com a nota final

df_resumo = spark.createDataFrame(
    [
        ("Estudo", "study_hours_per_day",
         df_gold.stat.corr("study_hours_per_day", "final_grade")),

        ("Frequência", "attendance_percentage",
         df_gold.stat.corr("attendance_percentage", "final_grade")),

        ("Foco", "focus_score",
         df_gold.stat.corr("focus_score", "final_grade")),

        ("Atividades", "assignments_completed",
         df_gold.stat.corr("assignments_completed", "final_grade")),

        ("Celular", "phone_usage_hours",
         df_gold.stat.corr("phone_usage_hours", "final_grade")),

        ("Redes sociais", "social_media_hours",
         df_gold.stat.corr("social_media_hours", "final_grade")),

        ("YouTube", "youtube_hours",
         df_gold.stat.corr("youtube_hours", "final_grade")),

        ("Games", "gaming_hours",
         df_gold.stat.corr("gaming_hours", "final_grade")),

        ("Entretenimento digital", "entertainment_hours",
         df_gold.stat.corr("entertainment_hours", "final_grade")),

        ("Sono", "sleep_hours",
         df_gold.stat.corr("sleep_hours", "final_grade")),

        ("Exercício", "exercise_minutes",
         df_gold.stat.corr("exercise_minutes", "final_grade"))
    ],
    ["dimensao", "variavel", "correlacao"]
)

df_resumo = (
    df_resumo
    .withColumn(
        "correlacao",
        F.round("correlacao", 4)
    )
)

display(df_resumo)

dimensao,variavel,correlacao
Estudo,study_hours_per_day,-0.0263
Frequência,attendance_percentage,-0.0188
Foco,focus_score,-0.0098
Atividades,assignments_completed,0.0154
Celular,phone_usage_hours,-0.0032
Redes sociais,social_media_hours,0.0016
YouTube,youtube_hours,-0.0379
Games,gaming_hours,0.0332
Entretenimento digital,entertainment_hours,-0.0011
Sono,sleep_hours,0.0217


In [0]:
# Análise descritiva por gênero

df_genero = (
    df_gold
    .groupBy("gender")
    .agg(
        F.count("*").alias("estudantes"),
        F.round(F.avg("final_grade"), 2).alias("nota_media"),
        F.round(F.avg("study_hours_per_day"), 2).alias("horas_estudo_media"),
        F.round(F.avg("attendance_percentage"), 2).alias("frequencia_media"),
        F.round(F.avg("focus_score"), 2).alias("foco_medio"),
        F.round(F.avg("entertainment_hours"), 2).alias("entretenimento_medio")
    )
    .orderBy("gender")
)

display(df_genero)

gender,estudantes,nota_media,horas_estudo_media,frequencia_media,foco_medio,entretenimento_medio
Female,2868,70.21,5.2,69.9,63.94,9.97
Male,2897,70.25,5.2,69.29,64.28,9.96
Other,234,69.71,5.03,68.79,65.3,9.95


In [0]:
GENDER_TABLE = "workspace.mvp_gold.performance_by_gender"

(
    df_genero.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GENDER_TABLE)
)

print("Tabela persistida:")
print(GENDER_TABLE)

Tabela persistida:
workspace.mvp_gold.performance_by_gender
